In [ ]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from riotwatcher import RiotWatcher, LolWatcher, ApiError

In [ ]:
load_dotenv(dotenv_path="../.env", override=True)
api_key = os.getenv("RIOT_API_KEY")

if not api_key:
    raise ValueError("RIOT_API_KEY not found in .env file")

riot_watcher = RiotWatcher(api_key)
lol_watcher = LolWatcher(api_key)

ACCOUNT_REGION = "asia"
PLATFORM_REGION = "kr"
MATCH_REGION = "asia"
MATCH_COUNT = 100

riot_id_name = "Hide on bush"
riot_id_tag = "KR1"

print("Watchers initialized successfully")

In [ ]:
account_data = riot_watcher.account.by_riot_id(ACCOUNT_REGION, riot_id_name, riot_id_tag)
puuid = account_data["puuid"]

print("PUUID fetched successfully")
print(puuid)

In [ ]:
match_ids = lol_watcher.match.matchlist_by_puuid(MATCH_REGION, puuid, count=MATCH_COUNT)
print("Total match ids fetched:", len(match_ids))
print(match_ids[:5])

In [ ]:
rows = []
failed_matches = []

for i, match_id in enumerate(match_ids, start=1):
    try:
        match_detail = lol_watcher.match.by_id(MATCH_REGION, match_id)
        info = match_detail["info"]
        participants = info["participants"]

        player_row = None
        for participant in participants:
            if participant["puuid"] == puuid:
                player_row = participant
                break

        if player_row is None:
            failed_matches.append(match_id)
            print(f"[{i}] Player not found in {match_id}")
            continue

        rows.append({
            "match_id": match_id,
            "game_creation": info.get("gameCreation"),
            "game_duration": info.get("gameDuration"),
            "game_mode": info.get("gameMode"),
            "queue_id": info.get("queueId"),
            "champion": player_row.get("championName"),
            "champ_level": player_row.get("champLevel"),
            "win": player_row.get("win"),
            "kills": player_row.get("kills"),
            "deaths": player_row.get("deaths"),
            "assists": player_row.get("assists"),
            "kda": (player_row.get("kills", 0) + player_row.get("assists", 0)) / max(player_row.get("deaths", 1), 1),
            "gold_earned": player_row.get("goldEarned"),
            "total_damage_to_champions": player_row.get("totalDamageDealtToChampions"),
            "total_damage_taken": player_row.get("totalDamageTaken"),
            "vision_score": player_row.get("visionScore"),
            "total_minions_killed": player_row.get("totalMinionsKilled"),
            "neutral_minions_killed": player_row.get("neutralMinionsKilled"),
            "wards_placed": player_row.get("wardsPlaced"),
            "wards_killed": player_row.get("wardsKilled"),
            "summoner1_id": player_row.get("summoner1Id"),
            "summoner2_id": player_row.get("summoner2Id"),
            "individual_position": player_row.get("individualPosition"),
            "team_position": player_row.get("teamPosition"),
        })

        print(f"[{i}] Success: {match_id}")
        time.sleep(1.2)

    except ApiError as err:
        failed_matches.append(match_id)
        print(f"[{i}] API error {err.response.status_code}: {match_id}")
        time.sleep(2)

In [ ]:
import csv
import os

print(type(rows))
print("Rows collected:", len(rows))

os.makedirs("../data/raw", exist_ok=True)

if len(rows) > 0:
    fieldnames = list(rows[0].keys())
    with open("../data/raw/faker_recent_matches.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print("Saved to ../data/raw/faker_recent_matches.csv")
else:
    print("No rows to save")

In [ ]:
import os
print(os.path.exists("../data/raw/faker_recent_matches.csv"))

In [ ]:
with open("../data/raw/faker_recent_matches.csv", "r", encoding="utf-8") as f:
    print(f.read())